#### Imports

In [1]:
# Imports
import os
import sys
import json
import yaml
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

#### Path Configurations

In [2]:
# Get the absolute path to the 'project' directory
project_root = os.path.abspath(os.path.join('..', '..'))

# Add it to sys.path if it's not already there
if project_root not in sys.path:
    sys.path.append(project_root)

# Path Configuration
data_directory = os.path.join(project_root , "data" , "detection")
dataset_name = "football-players-detection.v11i.yolov8"
dataset_yaml_path = os.path.join(data_directory , dataset_name , "data.yaml")

# Model Name and Weights
model_name = "27-04-2026_17-06_yolo26x"
model_directory = os.path.join(project_root , "models" , "detection", model_name)
full_model_weights_path = os.path.join(model_directory, "weights", "best.pt")

#### Model Setup

In [3]:
# Detection Model
detection_model = YOLO(full_model_weights_path)

#### Model Metrics

In [4]:
# Gathering Model Statistics
metrics = detection_model.val(
    data=dataset_yaml_path,
    imgsz=1280,
    save=False,
    plots=False,
    save_json=False,
)

Ultralytics 8.4.41 🚀 Python-3.12.3 torch-2.10.0+rocm7.2.0.gitb6ee5fde CUDA:0 (AMD Radeon RX 7800 XT, 16368MiB)
YOLO26x summary (fused): 190 layers, 55,638,168 parameters, 0 gradients, 193.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 142.8±19.8 MB/s, size: 180.6 KB)
val: Scanning /home/tom/Desktop/Programming/Personal/live-footie-formations/data/detection/football-players-detection.v11i.yolov8/valid/labels.cache... 49 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 49/49 17.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 8.5s/it 33.8s.4ss
                   all         49       1174      0.946      0.893      0.949      0.681
                  ball         45         45      0.939      0.733      0.862      0.469
            goalkeeper         38         39      0.899      0.914      0.954      0.729
                player         49        973      0.974      0.984      0.994      0.831
      

#### Gathering Additional Training Configs and Stats

In [5]:
# Paths for training arguments and results
model_training_args_path = os.path.join(model_directory,"args.yaml")
model_training_results_path = os.path.join(model_directory,"results.csv")

# Gathering training configuration
with open(model_training_args_path, "r") as f:
    training_config = yaml.safe_load(f)

# Gatheirng training results
training_results = pd.read_csv(model_training_results_path)
final_training_results = training_results.iloc[-1]

#### Per=class and Overall Stats

In [6]:
# Per-class stats
class_names = metrics.names
per_class_metrics = {}
for i, name in class_names.items():
    per_class_metrics[name] = {
        "precision": round(float(metrics.box.p[i]), 4),
        "recall":    round(float(metrics.box.r[i]), 4),
        "mAP50":     round(float(metrics.box.ap50[i]), 4),
        "mAP50-95":  round(float(metrics.box.ap[i]), 4),
    }

# Overall stats
overall_metrics = {
    "precision": round(float(metrics.box.mp), 4),
    "recall":    round(float(metrics.box.mr), 4),
    "mAP50":     round(float(metrics.box.map50), 4),
    "mAP50-95":  round(float(metrics.box.map), 4),
}

#### Full Metrics

In [7]:
# Full Metrics
metrics_to_save = {
    'model_name': model_name,
    'pretrained_base_model': training_config['model'],
    'training_dataset': Path(training_config['data']).parent.name,
    'evaluation_dataset': dataset_name,
    'imgsz': training_config['imgsz'],
    'intended_epochs_trained': training_config['epochs'],
    'epochs_trained': final_training_results['epoch'],
    'overall_metrics': overall_metrics,
    'per_class_overall_metrics':per_class_metrics,
    'total_training_time': final_training_results['time'],
    'ending_training_metrics' : {
        "precision": final_training_results['metrics/precision(B)'],
        "recall":    final_training_results['metrics/recall(B)'],
        "mAP50":     final_training_results['metrics/mAP50(B)'],
        "mAP50-95":  final_training_results['metrics/mAP50-95(B)'],
        },
    'training_args': training_config
    }

#### Saving JSON Metrics

In [8]:
full_model_eval_results_path = os.path.join(model_directory, f'{model_name}_{dataset_name}_results.json')
with open(full_model_eval_results_path, "w") as f:
    json.dump(metrics_to_save, f, indent=4)